# SA 2 - Regresión logística y árbol de decisión

## Justificación

Comparo regresión logística y árbol de decisión. La logística entrega probabilidades y el árbol permite revisar sus reglas.

## Diseño del modelo

El proceso está ordenado en celdas: carga, preparación, ajuste, evaluación y guardado del modelo.

Repositorio: https://github.com/axelisaak10/evaluacion-2

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

ruta = Path("..")
datos = pd.read_csv(ruta / "data" / "breast-cancer.csv")
datos["diagnosis"].value_counts()

## Preparación

In [ ]:
x = datos.drop(columns=["id", "diagnosis"])
y = (datos["diagnosis"] == "M").astype(int)

x_entrena, x_prueba, y_entrena, y_prueba = train_test_split(
    x,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Regresión logística

In [ ]:
logistica = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=10000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42,
    )),
])

buscar_logistica = GridSearchCV(
    logistica,
    {
        "modelo__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "modelo__penalty": ["l1", "l2"],
    },
    cv=validacion,
    scoring="roc_auc",
)
buscar_logistica.fit(x_entrena, y_entrena)
buscar_logistica.best_params_

## Árbol de decisión

In [ ]:
arbol = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42,
)

buscar_arbol = GridSearchCV(
    arbol,
    {
        "criterion": ["gini", "entropy"],
        "max_depth": [2, 3, 4, 5, None],
        "min_samples_split": [2, 10],
        "min_samples_leaf": [1, 4, 8],
        "ccp_alpha": [0, 0.005, 0.01],
    },
    cv=validacion,
    scoring="roc_auc",
)
buscar_arbol.fit(x_entrena, y_entrena)
buscar_arbol.best_params_

## Evaluación y optimización

In [ ]:
pred_logistica = buscar_logistica.predict(x_prueba)
prob_logistica = buscar_logistica.predict_proba(x_prueba)[:, 1]

pred_arbol = buscar_arbol.predict(x_prueba)
prob_arbol = buscar_arbol.predict_proba(x_prueba)[:, 1]

resultados = pd.DataFrame({
    "modelo": ["Logística", "Árbol"],
    "accuracy": [
        accuracy_score(y_prueba, pred_logistica),
        accuracy_score(y_prueba, pred_arbol),
    ],
    "precision": [
        precision_score(y_prueba, pred_logistica),
        precision_score(y_prueba, pred_arbol),
    ],
    "recall": [
        recall_score(y_prueba, pred_logistica),
        recall_score(y_prueba, pred_arbol),
    ],
    "f1": [
        f1_score(y_prueba, pred_logistica),
        f1_score(y_prueba, pred_arbol),
    ],
    "AUC": [
        roc_auc_score(y_prueba, prob_logistica),
        roc_auc_score(y_prueba, prob_arbol),
    ],
})
resultados

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(confusion_matrix(y_prueba, pred_logistica), annot=True, fmt="d", ax=ejes[0])
sns.heatmap(confusion_matrix(y_prueba, pred_arbol), annot=True, fmt="d", ax=ejes[1])
ejes[0].set_title("Logística")
ejes[1].set_title("Árbol")

for grafica in ejes:
    grafica.set_xlabel("Predicción")
    grafica.set_ylabel("Real")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_prueba, prob_logistica, name="Logística")
RocCurveDisplay.from_predictions(y_prueba, prob_arbol, name="Árbol", ax=plt.gca())
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("Comparación ROC")
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    buscar_arbol.best_estimator_,
    feature_names=x.columns,
    class_names=["Benigno", "Maligno"],
    filled=True,
    rounded=True,
    fontsize=7,
)
plt.title("Árbol de decisión")
plt.show()

In [ ]:
joblib.dump(buscar_logistica.best_estimator_, ruta / "models" / "sa_logistica_breast_cancer.joblib")
joblib.dump(buscar_arbol.best_estimator_, ruta / "models" / "sa_arbol_breast_cancer.joblib")

## Interpretación

La regresión logística tuvo mejor AUC y recall en la prueba. El árbol es menos preciso, pero permite ver las variables y cortes usados para clasificar.